# 🩺 SaúdeMap IA — Treinamento da IA de Triagem

**Objetivo:** classificar a descrição de um sintoma (texto livre, em português) na especialidade médica correta — ou identificar urgência.

**Classes:** Odontologia · Pediatria · Clínico Geral · Cardiologia · Ginecologia · Urgência · Fora do escopo

**Técnica:** TF-IDF (palavras + pares + n-gramas de letras) → Regressão Logística, com duas camadas de segurança:
1. **Sinais de alarme** (regras inspiradas em protocolos de triagem) → urgência
2. **Viés de segurança**: probabilidade de urgência ≥ 25% → urgência

**Avaliação honesta:** validação cruzada *agrupada* + três conjuntos de teste escritos à parte. Só o **teste C** ficou cego até o fim — é o número que vale.

> Execute: **Ambiente de execução → Executar tudo**.

## 1. Gerar o dataset de treino
Frases-base escritas à mão + variações (gírias, erros de digitação). A coluna `grupo` guarda o id da frase-base: variações da mesma frase nunca ficam em treino e validação ao mesmo tempo.

In [ ]:
%%writefile gerar_dataset.py
"""
gerar_dataset.py — Construção do corpus de treino da IA de triagem (SaúdeMap IA)

Estratégia:
  1. Frases-base escritas à mão por categoria (sintomas + sujeitos + contextos)
  2. Combinação controlada sujeito × sintoma × complemento
  3. Aumentação: gírias ("tô", "tá", "mt"), erros de digitação, complementos de tempo
  4. Cada frase carrega um "grupo" (id da frase-base) — usado na validação cruzada
     agrupada, para que variações da MESMA frase nunca caiam em treino e validação
     ao mesmo tempo (isso inflaria artificialmente a acurácia).

Critério clínico adotado (segurança do paciente):
  - Dor no peito LEVE/crônica, pressão, colesterol, palpitação  -> cardio
  - Dor no peito FORTE/súbita, com suor, irradiando pro braço,
    falta de ar em repouso, sinais de AVC, desmaio, sangramento  -> urgencia
  - Na dúvida entre cardio e urgência, a regra de segurança do
    classificador favorece urgência (ver treinar_modelo.py).
"""
import csv
import random

random.seed(42)

# ───────────────────────────── DENTISTA ─────────────────────────────
DENT_SINT = [
    "dor de dente", "dor no dente", "dente doendo", "dor forte no dente", "dente latejando",
    "gengiva inflamada", "gengiva sangrando", "gengiva inchada", "sangramento na gengiva",
    "dente quebrado", "dente lascado", "dente mole", "dente trincado", "carie no dente",
    "buraco no dente", "dente furado", "dente sensivel ao gelado", "sensibilidade nos dentes",
    "dor ao mastigar", "siso nascendo", "siso inflamado", "dente do siso doendo",
    "abscesso no dente", "rosto inchado por causa do dente", "canal inflamado",
    "restauracao caiu", "obturacao caiu", "aparelho machucando a boca", "aparelho ortodontico soltou",
    "dentadura machucando", "protese dentaria solta", "mau halito forte", "tartaro nos dentes",
    "dente escurecido", "dor no dente quando bebo agua gelada", "dente de leite mole doendo",
    "dor de dente que nao passa com remedio", "pus na gengiva", "bolinha na gengiva",
    "dor na gengiva perto do siso", "dente amarelado e dolorido", "gengiva retraida",
]
DENT_EXTRA = [
    "preciso fazer limpeza nos dentes", "quero arrancar um dente", "preciso extrair um dente",
    "preciso fazer canal", "quero consultar um dentista", "avaliacao odontologica",
    "tratamento dentario", "quero fazer restauracao", "consulta com dentista para meu filho",
]
DENT_SUJ = ["", "estou com", "to com", "tenho", "sinto", "estou sentindo", "acordei com",
            "meu filho esta com", "minha mae esta com", "meu marido esta com"]

# ───────────────────────────── PEDIATRA ─────────────────────────────
PED_SUJ = ["meu bebe", "minha bebe", "meu filho", "minha filha", "meu nenem", "a crianca",
           "meu filho de 2 anos", "minha filha de 4 anos", "meu menino", "minha menina",
           "o recem nascido", "meu bebe de 6 meses", "minha crianca", "meu filho pequeno",
           "o nenem", "meu sobrinho de 3 anos"]
PED_VERBO = ["esta com", "ta com", "anda com", "acordou com", "tem", "apareceu com"]
PED_SINT = [
    "febre", "febre alta", "tosse", "tosse com catarro", "catarro", "nariz escorrendo", "coriza",
    "diarreia", "vomito", "manchas vermelhas na pele", "bolinhas pelo corpo", "catapora",
    "assadura", "dor de ouvido", "dor de barriga", "colica", "garganta inflamada", "piolho",
    "lombriga", "alergia na pele", "brotoeja", "chiado no peito", "olho vermelho", "conjuntivite",
    "pouco apetite", "muita moleza", "ranho", "gripe", "resfriado", "sapinho na boca",
    "caxumba", "dor de garganta", "virose", "febre e tosse", "diarreia e vomito",
]
PED_FRASES = [
    "meu bebe nao quer mamar", "o bebe chora sem parar", "minha filha nao quer comer nada",
    "consulta de rotina do bebe", "acompanhamento do crescimento do meu filho",
    "pesar e medir o bebe", "puericultura do recem nascido", "as vacinas da crianca estao atrasadas",
    "teste do pezinho", "meu filho nao esta ganhando peso", "o nenem esta vomitando depois de mamar",
    "meu filho reclama de dor na barriga toda manha", "minha filha coca muito a cabeca",
    "o bebe esta com o umbigo vermelho", "meu filho molha a cama a noite",
    "preciso de pediatra para meu filho", "consulta com pediatra", "minha crianca vive gripada",
    "o bebe dorme demais e esta molinho", "meu filho tem bronquite",
]

# ───────────────────────────── CLÍNICO ──────────────────────────────
CLIN_SUJ = ["", "estou com", "to com", "eu estou com", "sinto", "tenho", "acordei com",
            "estou sentindo", "meu marido esta com", "minha esposa esta com", "minha mae esta com",
            "meu pai esta com", "minha avo esta com", "meu irmao esta com", "faz dias que estou com"]
CLIN_SINT = [
    "febre", "gripe", "resfriado", "tosse seca", "tosse com catarro", "dor de garganta",
    "garganta inflamada", "nariz entupido", "sinusite", "dor de cabeca", "enxaqueca",
    "dor no corpo", "dor nas costas", "dor lombar", "dor na coluna", "dor de barriga",
    "diarreia", "vomito", "enjoo", "azia", "gastrite", "queimacao no estomago",
    "ardencia ao urinar", "infeccao urinaria", "dor ao fazer xixi", "tontura", "cansaco",
    "fraqueza", "insonia", "dor no joelho", "dor no ombro", "dor muscular", "alergia",
    "coceira no corpo", "micose", "unha encravada", "dor de ouvido", "conjuntivite",
    "glicose alta", "anemia", "prisao de ventre", "hemorroida", "dor nas juntas",
    "febre e dor no corpo", "mal estar", "dor no pe", "espinha inflamada", "virose",
    "tosse que nao passa", "catarro no peito", "dor no punho", "labirintite", "rinite",
]
CLIN_FRASES = [
    "preciso renovar minha receita", "quero pedir exame de sangue", "quero fazer um check up",
    "exame de rotina", "preciso de atestado medico", "consulta de rotina", "preciso de encaminhamento para especialista",
    "quero mostrar resultado de exame", "preciso de receita do remedio de diabetes",
    "quero consultar com clinico geral", "tenho diabetes e preciso de acompanhamento",
    "meu remedio de tireoide acabou", "quero fazer exame de urina", "preciso trocar meu remedio",
    "quero avaliar minha saude em geral", "estou me sentindo fraco e sem energia",
    "estou emagrecendo sem motivo", "machuquei o pe jogando bola", "torci o tornozelo",
    "cortei o dedo e esta inflamado", "fui picado por inseto e inchou", "estou com a imunidade baixa",
]

# ───────────────────────────── CARDIO ───────────────────────────────
CARD_SUJ = ["", "tenho", "estou com", "sinto", "meu pai tem", "minha mae tem", "meu avo tem",
            "estou tendo", "ando com", "faz tempo que tenho"]
CARD_SINT = [
    "pressao alta", "hipertensao", "pressao descontrolada", "pressao subindo toda hora",
    "colesterol alto", "triglicerides alto", "palpitacao", "coracao acelerado",
    "coracao disparando as vezes", "batedeira no coracao", "arritmia", "coracao batendo fora do ritmo",
    "cansaco ao subir escada", "falta de ar quando faco esforco", "inchaco nos pes e tornozelos",
    "pernas inchadas no fim do dia", "sopro no coracao", "dor leve no peito quando faco esforco",
    "aperto leve no peito quando caminho rapido", "coracao grande", "insuficiencia cardiaca",
    "pressao alta mesmo tomando remedio", "tontura quando a pressao sobe",
]
CARD_FRASES = [
    "quero fazer eletrocardiograma", "preciso fazer ecocardiograma", "quero fazer teste ergometrico",
    "tenho historico de infarto na familia", "ja tive infarto e preciso de acompanhamento",
    "uso marcapasso e preciso de revisao", "quero fazer um check up do coracao",
    "preciso de avaliacao cardiologica", "quero saber meu risco cardiaco",
    "preciso de cardiologista", "consulta com cardiologista", "controle da pressao arterial",
    "meu remedio de pressao nao esta fazendo efeito", "quero medir minha pressao",
    "o medico pediu exame do coracao", "tenho colesterol alto e quero tratar",
    "sinto o coracao pular uma batida de vez em quando", "meu coracao acelera quando deito",
]

# ───────────────────────────── GINECO ───────────────────────────────
GIN_SUJ = ["", "estou com", "tenho", "to com", "sinto", "estou tendo", "minha esposa esta com",
           "minha filha adolescente esta com", "faz um mes que estou com"]
GIN_SINT = [
    "atraso menstrual", "menstruacao atrasada", "menstruacao irregular", "colica menstrual forte",
    "menstruacao muito forte", "sangramento fora da menstruacao", "corrimento", "corrimento com cheiro forte",
    "coceira intima", "ardencia intima", "candidiase", "dor na relacao", "caroco no seio",
    "nodulo na mama", "dor no seio", "calorao da menopausa", "ovario policistico", "mioma",
    "infeccao vaginal", "endometriose", "tpm muito forte", "corrimento amarelado",
    "menstruacao que nao para", "sangramento depois da relacao", "dor no pe da barriga na menstruacao",
    "coceira na vagina", "secrecao no bico do seio",
]
GIN_FRASES = [
    "quero fazer exame preventivo", "preciso fazer papanicolau", "consulta ginecologica",
    "quero comecar a tomar anticoncepcional", "quero trocar de anticoncepcional", "quero colocar diu",
    "quero fazer teste de gravidez", "acho que estou gravida", "preciso fazer pre natal",
    "estou gravida e preciso de acompanhamento", "quero fazer mamografia", "estou entrando na menopausa",
    "orientacao sobre planejamento familiar", "preciso de ginecologista", "consulta com ginecologista",
    "a menstruacao nao desceu esse mes", "minha menstruacao atrasou duas semanas",
    "gravida de 3 meses e quero acompanhamento", "quero fazer ultrassom transvaginal",
]

# ───────────────────────────── URGÊNCIA ─────────────────────────────
URG_FRASES = [
    "dor no peito muito forte", "dor no peito que nao passa", "aperto no peito com suor frio",
    "dor no peito que vai para o braco esquerdo", "dor no peito e braco formigando",
    "falta de ar muito forte", "nao consigo respirar", "estou sufocando", "ele desmaiou",
    "desmaiei agora", "perdeu a consciencia", "ela nao acorda", "esta tendo convulsao",
    "teve uma convulsao", "ficou com a boca torta de repente", "um lado do corpo paralisou",
    "fala enrolada de repente", "perdeu a forca no braco de repente", "sangramento que nao para",
    "esta sangrando muito", "vomitando sangue", "corte profundo sangrando", "sofri acidente de moto",
    "acidente de carro grave", "foi atropelado", "caiu do telhado", "bateu a cabeca muito forte",
    "fratura exposta", "osso aparecendo na perna", "queimadura grave", "queimou com oleo quente",
    "engoliu veneno", "tomou remedio demais", "crianca engoliu produto de limpeza",
    "picada de cobra", "picada de escorpiao", "reacao alergica grave", "garganta fechando",
    "rosto e labios inchando depois do remedio", "engasgado sem conseguir respirar", "quase se afogou",
    "levou choque eletrico", "bebe com febre muito alta e convulsao", "bebe ficou roxo",
    "bebe nao esta respirando direito", "gravida sangrando muito", "a bolsa estourou",
    "contracoes muito fortes gravida", "dor de cabeca subita muito forte a pior da vida",
    "desmaiou e bateu a cabeca", "labios roxos e falta de ar", "suor frio e dor no peito",
    "pressao muito alta com dor no peito e visao turva", "confusao mental de repente",
    "levou uma facada", "levou um tiro", "sangue saindo pelo ouvido depois da queda",
    "dor na barriga insuportavel e barriga dura", "febre alta com manchas roxas no corpo",
    "infarto", "acho que estou tendo um infarto", "acho que ele esta tendo um avc", "avc",
    "parada cardiaca", "desmaio com dor no peito",
]
URG_PREF = ["", "", "", "socorro ", "urgente ", "me ajuda ", "rapido ", "emergencia "]
URG_SUF = ["", "", "", " agora", " o que eu faco", " de repente", " rapido"]

# ───────────────────────────── OUTRO ────────────────────────────────
OUTRO_FRASES = [
    "bom dia", "boa tarde", "boa noite", "oi", "ola", "obrigado", "obrigada", "valeu", "teste",
    "testando", "qual o horario de funcionamento", "que horas abre o posto", "que horas fecha a ubs",
    "onde fica o posto de saude", "endereco da ubs centro", "telefone do posto", "como faco para marcar consulta",
    "como agendar atendimento", "como tirar cartao do sus", "quais vacinas estao disponiveis",
    "quando comeca a campanha de vacinacao", "tem vacina da gripe", "o posto abre sabado",
    "quem e o medico de hoje", "lista de medicos", "como funciona o site", "esqueci minha senha",
    "quero trocar minha senha", "como sair da conta", "asdfgh", "kkkkkk", "aaaaa", "qualquer coisa",
    "nada", "sim", "nao", "ok", "futebol", "receita de bolo", "previsao do tempo",
    "quero falar com a secretaria de saude", "quero fazer uma reclamacao", "quero fazer um elogio",
    "onde fica a farmacia popular", "como chegar no hospital municipal", "o posto atende hoje",
    "precisa de agendamento", "quanto custa a consulta", "aceita cartao do sus",
    "qual o numero do samu", "horario da vacinacao", "posto mais perto de mim", "mapa dos postos",
    "calendario de vacinacao", "noticias da saude", "quem desenvolveu esse site", "como me cadastrar",
    "onde vejo meu historico de vacinas", "tem estacionamento no posto", "posso levar acompanhante",
    "hello", "tudo bem", "quero saber sobre a campanha de hpv", "que dia e hoje", "abc123",
    "o site esta lento", "nao consigo entrar", "atualizar meus dados", "fila do posto esta grande",
]
OUTRO_PREF = ["", "", "por favor ", "gostaria de saber ", "me diz ", "uma pergunta "]


# ─────────── Lacunas de cobertura identificadas na 1ª avaliação ───────────
# (a) Emergência com CRIANÇA deve ser urgência, não pediatria
URG_CRIANCA_SUJ = ["meu filho", "minha filha", "o bebe", "a crianca", "meu nenem", "minha bebe",
                   "meu filho pequeno", "a nenem"]
URG_CRIANCA_EVT = ["engoliu produto de limpeza", "bebeu agua sanitaria", "tomou veneno", "ficou roxo",
                   "nao esta respirando", "esta engasgado", "esta convulsionando", "caiu e bateu a cabeca forte",
                   "desmaiou", "nao acorda", "se queimou feio", "engoliu uma pilha", "bebeu querosene",
                   "esta com os labios roxos", "tomou os remedios da avo", "caiu na piscina",
                   "esta mole e nao reage", "engoliu uma bala e sufocou"]
# (b) Picadas, mordidas e envenenamentos
URG_ANIMAL = ["picado por escorpiao", "escorpiao me picou", "mordida de cobra", "cobra me mordeu",
              "picada de aranha marrom", "abelha picou e estou inchando todo", "fui picado por varias abelhas",
              "cachorro mordeu e esta sangrando muito", "comi planta venenosa", "bebi veneno por engano"]
# (c) "nao consigo X" que NÃO é urgência
CLIN_NAO_CONSIGO = ["tenho dificuldade para dormir", "nao consigo pegar no sono", "acordo varias vezes a noite",
                    "nao consigo parar de espirrar", "nao consigo emagrecer", "nao durmo bem faz tempo"]
# (d) Frases de agradecimento / pedido de ajuda com o site
OUTRO_AJUDA = ["muito obrigado", "agradeco a ajuda", "preciso de ajuda com o cadastro", "ajuda para usar o site",
               "obrigado pela atencao", "valeu pela informacao", "me ajuda a achar o posto", "ajuda com a senha"]
# (e) "peito" no sentido de mama e sangramento ginecológico comum
GIN_EXTRA = ["caroco no peito", "bolinha no seio", "nodulo no peito", "achei uma bolinha na mama",
             "sangramento fora do periodo", "sangrando fora do ciclo", "escape menstrual",
             "sangramento entre as menstruacoes", "peito dolorido perto da menstruacao"]
# (f) Clínico com palavras que colidiam com cardio ("quando ando", "faco")
CLIN_EXTRA = ["arde para urinar", "xixi ardendo", "dor ao urinar", "urina com ardor", "dor no joelho quando caminho",
              "dor na perna quando ando", "dor no quadril ao andar", "garganta doendo", "doi a garganta para engolir",
              "check up geral com pressao normal", "exame geral de saude"]
DENT_EXTRA2 = ["acho que tenho carie", "carie doendo", "carie no dente de tras"]
CARD_EXTRA = ["fazer eletro", "exame eletro do coracao", "pedir um eletro", "fazer eco do coracao"]

# ───────────────────── Aumentação (gírias e erros) ─────────────────────
GIRIAS = [(" esta ", " ta "), (" estou ", " to "), (" muito ", " mt "), (" para ", " pra "),
          (" voce ", " vc "), (" porque ", " pq "), (" nao ", " n "), (" tambem ", " tb ")]
TEMPO = ["", "", "", " desde ontem", " faz tres dias", " ha uma semana", " desde cedo",
         " faz dias", " toda noite"]
EDUC = ["", "", "", "", "doutor ", "por favor ", "boa tarde ", "oi "]


def gírias(t):
    t = f" {t} "
    for a, b in GIRIAS:
        if a in t and random.random() < 0.5:
            t = t.replace(a, b)
    return t.strip()


def erro_digitacao(t):
    palavras = t.split()
    cand = [i for i, w in enumerate(palavras) if len(w) >= 5]
    if not cand:
        return t
    i = random.choice(cand)
    w = palavras[i]
    j = random.randrange(1, len(w) - 1)
    op = random.choice(["apaga", "duplica", "troca"])
    if op == "apaga":
        w = w[:j] + w[j + 1:]
    elif op == "duplica":
        w = w[:j] + w[j] + w[j:]
    else:
        w = w[:j - 1] + w[j] + w[j - 1] + w[j + 1:]
    palavras[i] = w
    return " ".join(palavras)


def variar(base, n):
    """Gera n variações de uma frase-base (a própria base incluída)."""
    out = {base}
    tentativas = 0
    while len(out) < n and tentativas < n * 10:
        tentativas += 1
        t = base
        if random.random() < 0.35:
            t = random.choice(EDUC) + t
        if random.random() < 0.35:
            t = t + random.choice(TEMPO)
        t = gírias(t)
        if random.random() < 0.30:
            t = erro_digitacao(t)
        out.add(" ".join(t.split()))
    return list(out)


def combinar(sujeitos, sintomas):
    frases = []
    for s in sintomas:
        for suj in random.sample(sujeitos, min(4, len(sujeitos))):
            frases.append(f"{suj} {s}".strip())
    return frases


linhas = []
grupo_id = 0


def adicionar(categoria, bases, n_var):
    global grupo_id
    for b in bases:
        grupo_id += 1
        for v in variar(b, n_var):
            linhas.append((v, categoria, grupo_id))


adicionar("dentista", combinar(DENT_SUJ, DENT_SINT) + DENT_EXTRA, 3)
ped_bases = []
for s in PED_SINT:
    for suj in random.sample(PED_SUJ, 4):
        ped_bases.append(f"{suj} {random.choice(PED_VERBO)} {s}")
adicionar("pediatra", ped_bases + PED_FRASES, 3)
adicionar("clinico", combinar(CLIN_SUJ, CLIN_SINT) + CLIN_FRASES, 2)
adicionar("cardio", combinar(CARD_SUJ, CARD_SINT) + CARD_FRASES, 4)
adicionar("gineco", combinar(GIN_SUJ, GIN_SINT) + GIN_FRASES, 3)

urg_bases = []
for f in URG_FRASES:
    for _ in range(3):
        urg_bases.append((random.choice(URG_PREF) + f + random.choice(URG_SUF)).strip())
adicionar("urgencia", list(dict.fromkeys(urg_bases)), 3)

outro_bases = list(dict.fromkeys((random.choice(OUTRO_PREF) + f).strip()
                                 for f in OUTRO_FRASES for _ in range(3)))
adicionar("outro", outro_bases, 3)


urg_cri = [f"{s} {e}" for s in URG_CRIANCA_SUJ for e in random.sample(URG_CRIANCA_EVT, 6)]
adicionar("urgencia", urg_cri + URG_ANIMAL, 3)
adicionar("clinico", combinar(CLIN_SUJ, CLIN_EXTRA) + CLIN_NAO_CONSIGO, 3)
adicionar("outro", OUTRO_AJUDA, 3)
adicionar("gineco", combinar(GIN_SUJ, GIN_EXTRA), 3)
adicionar("dentista", DENT_EXTRA2, 3)
adicionar("cardio", CARD_EXTRA, 3)

# remove duplicatas exatas mantendo a primeira ocorrência
vistos, final = set(), []
for t, c, g in linhas:
    if t not in vistos:
        vistos.add(t)
        final.append((t, c, g))

with open("dataset_treino.csv", "w", newline="", encoding="utf-8") as f:
    w = csv.writer(f)
    w.writerow(["texto", "categoria", "grupo"])
    w.writerows(final)

from collections import Counter
print(f"Total: {len(final)} frases | {grupo_id} frases-base")
for c, n in sorted(Counter(c for _, c, _ in final).items()):
    print(f"  {c:10s} {n}")


In [ ]:
!python gerar_dataset.py

## 2. Conjuntos de teste
Baixa do GitHub (se o repositório for público) ou pede upload de `dataset_teste.csv`, `dataset_teste_b.csv` e `dataset_teste_c.csv` (pasta `ia/`).

In [ ]:
import os, urllib.request
REPO = "https://raw.githubusercontent.com/thzz099/saude-acessivel/main/ia/"
arquivos = ["dataset_teste.csv", "dataset_teste_b.csv", "dataset_teste_c.csv"]
faltando = []
for a in arquivos:
    if os.path.exists(a):
        continue
    try:
        urllib.request.urlretrieve(REPO + a, a)
        print("baixado:", a)
    except Exception:
        faltando.append(a)
if faltando:
    print("Faça upload de:", faltando)
    from google.colab import files
    files.upload()
print("Prontos:", [a for a in arquivos if os.path.exists(a)])

## 3. Treinar, avaliar e exportar
Compara 4 algoritmos, treina o final, avalia nos testes A, B e C, gera a matriz de confusão e exporta `modelo-ia.js`.

In [ ]:
%%writefile treinar_modelo.py
"""
treinar_modelo.py — Treino, avaliação e exportação da IA de triagem (SaúdeMap IA)

Roda igual localmente ou no Google Colab. Saídas:
  - modelo-ia.js              pesos do modelo, carregado pelo site
  - relatorio_metricas.json   números da avaliação
  - matriz_confusao.png       matriz de confusão no conjunto de teste
"""
import csv
import json
import math
import unicodedata
from collections import Counter

import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.naive_bayes import ComplementNB
from sklearn.svm import LinearSVC

# ═════════════ PRÉ-PROCESSAMENTO (replicado idêntico no JavaScript) ═════════════
STOPWORDS = {
    "de", "da", "do", "das", "dos", "a", "o", "as", "os", "e", "em", "no", "na", "nos", "nas",
    "um", "uma", "uns", "umas", "com", "por", "pra", "para", "que", "eu", "me", "meu", "minha",
    "meus", "minhas", "se", "ja", "la", "aqui", "isso", "isto", "esse", "essa", "ele", "ela",
}


def normalizar(texto):
    t = unicodedata.normalize("NFD", texto.lower())
    t = "".join(ch for ch in t if not ("\u0300" <= ch <= "\u036f"))
    t = "".join(ch if ("a" <= ch <= "z" or "0" <= ch <= "9") else " " for ch in t)
    return t.split()


def analisar(texto):
    """Palavras + pares de palavras + trigramas/quadrigramas de caracteres.
    Os n-gramas de caracteres dão robustez a erro de digitação ('dentee', 'febr')."""
    palavras = [w for w in normalizar(texto) if w not in STOPWORDS]
    feats = ["w:" + w for w in palavras]
    feats += ["b:" + palavras[i] + "_" + palavras[i + 1] for i in range(len(palavras) - 1)]
    for w in palavras:
        p = " " + w + " "
        for n in (3, 4):
            for i in range(len(p) - n + 1):
                feats.append("c:" + p[i:i + n])
    return feats



# ═════════════ SINAIS DE ALARME (camada de regras sobre o modelo) ═════════════
# Inspirado nos discriminadores de protocolos de triagem (ex.: Manchester):
# se qualquer sinal aparecer, é URGÊNCIA — independente do que o modelo disse.
# Aplicados ao texto normalizado (minúsculo, sem acento). Sintaxe compatível
# com RegExp do JavaScript: o MESMO conjunto é exportado para o site.
SINAIS_ALARME = [
    r"\bdesmai", r"\bdesacordad", r"\binconscien", r"\bnao (acorda|reage|responde)\b",
    r"\bperdeu a consciencia", r"\bconvuls",
    r"\bnao\b[a-z ]{0,25}\brespir", r"\bparou de respirar", r"\bsufoc", r"\bengasg",
    r"\binfarto", r"\bavc\b", r"\bderrame\b", r"\bparada cardiaca",
    r"\bboca torta", r"\bfala (enrolad|embolad)", r"\b(lado do corpo|braco) (paralis|mole|dormente)",
    r"\bacidente\b", r"\batropelad", r"\bbateram o carro", r"\bbatida de (carro|moto)", r"\bcapot",
    r"\bfacada", r"\besfaquead", r"\btiro\b", r"\bbalead",
    r"\bsangr[a-z]* (muito|demais|sem parar)", r"\bsangue nao (para|estanca)", r"\bhemorragia",
    r"\bvomitando sangue",
    r"\bveneno", r"\benvenen", r"\bagua sanitaria", r"\bsoda caustica", r"\bquerosene", r"\boverdose",
    r"\btomou (muito|muitos|um monte de|todos os|remedio de adulto)", r"\bremedio[a-z ]{0,20}por engano",
    r"\b(picad|mordid)[oa]s? (de|por) (uma )?(cobra|escorpiao|aranha)", r"\bcobra (me )?mord",
    r"\bescorpiao (me )?pic",
    r"\bqueimadura (grave|no corpo)", r"\bqueim[a-z]* (com oleo|com agua fervendo|feio)",
    r"\b(bebe|labios|boca) (ficou |esta |ta )?rox", r"\bficou rox",
    r"\bbolsa (estourou|rompeu)", r"\bestourou a bolsa",
    r"\bdor no peito (muito forte|forte|fortissima|apertando|insuportavel)", r"\baperto no peito com suor",
    r"\bgarganta fechando", r"\bcaiu (de|do|da) (altura|telhado|laje|escada|moto)", r"\bbateu a cabeca",
]
# Crise de saúde mental: não vai para a UPA — mostra CVV 188 e SAMU 192.
SINAIS_CRISE = [
    r"\bme matar", r"\bsuicid", r"\btirar (a )?minha (propria )?vida", r"\bnao quero mais viver",
    r"\bquero morrer", r"\bacabar com (a )?minha vida", r"\bme (cortar|machucar)\b",
]
import re
_RE_ALARME = [re.compile(x) for x in SINAIS_ALARME]


def sinal_alarme(texto):
    t = " ".join(normalizar(texto))
    return any(r.search(t) for r in _RE_ALARME)


def novo_vetorizador():
    return TfidfVectorizer(analyzer=analisar, sublinear_tf=True, min_df=2, norm="l2")


# ═════════════ DADOS ═════════════
def ler(caminho, com_grupo):
    with open(caminho, encoding="utf-8") as f:
        r = list(csv.DictReader(f))
    X = [x["texto"] for x in r]
    y = [x["categoria"] for x in r]
    g = [int(x["grupo"]) for x in r] if com_grupo else None
    return X, y, g


X_tr, y_tr, g_tr = ler("dataset_treino.csv", True)
X_te, y_te, _ = ler("dataset_teste.csv", False)
X_te_orig = list(X_te)

# Vazamento: nenhuma frase de teste pode existir no treino (após normalização)
norm_tr = {" ".join(normalizar(t)) for t in X_tr}
vazadas = [t for t in X_te if " ".join(normalizar(t)) in norm_tr]
print(f"Treino: {len(X_tr)} frases | Teste: {len(X_te)} frases | Vazamento treino→teste: {len(vazadas)}")
if vazadas:
    print("  ATENÇÃO — frases de teste presentes no treino:", vazadas)

CLASSES = sorted(set(y_tr))
NOMES = {"cardio": "Cardiologia", "clinico": "Clínico Geral", "dentista": "Odontologia",
         "gineco": "Ginecologia", "pediatra": "Pediatria", "urgencia": "Urgência / Emergência",
         "outro": "Fora do escopo"}

# ═════════════ COMPARAÇÃO DE ALGORITMOS (validação cruzada AGRUPADA) ═════════════
# StratifiedGroupKFold: variações da mesma frase-base ficam SEMPRE no mesmo fold.
cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)
candidatos = {
    "Regressão Logística (C=5)": lambda: LogisticRegression(C=5, max_iter=3000),
    "Regressão Logística (C=20)": lambda: LogisticRegression(C=20, max_iter=3000),
    "SVM Linear": lambda: LinearSVC(C=1.0),
    "Naive Bayes (Complement)": lambda: ComplementNB(alpha=0.3),
}
Xa, ya, ga = np.array(X_tr, dtype=object), np.array(y_tr), np.array(g_tr)
resultados_cv = {}
print("\nValidação cruzada agrupada (5 folds):")
for nome, fabrica in candidatos.items():
    accs, f1s = [], []
    for tr_idx, va_idx in cv.split(Xa, ya, ga):
        vec = novo_vetorizador()
        Xtr_v = vec.fit_transform(Xa[tr_idx])
        mdl = fabrica().fit(Xtr_v, ya[tr_idx])
        pred = mdl.predict(vec.transform(Xa[va_idx]))
        accs.append(accuracy_score(ya[va_idx], pred))
        f1s.append(f1_score(ya[va_idx], pred, average="macro"))
    resultados_cv[nome] = {"acuracia_media": float(np.mean(accs)), "acuracia_desvio": float(np.std(accs)),
                           "f1_macro_media": float(np.mean(f1s))}
    print(f"  {nome:28s} acurácia {np.mean(accs):.3f} ± {np.std(accs):.3f} | F1 macro {np.mean(f1s):.3f}")

# ═════════════ MODELO FINAL ═════════════
# Escolha: Regressão Logística — produz PROBABILIDADES calibradas (necessárias para
# a regra de abstenção e a regra de segurança de urgência) e é exportável como
# uma simples multiplicação de matrizes no JavaScript.
C_FINAL = 20 if resultados_cv["Regressão Logística (C=20)"]["f1_macro_media"] >= \
               resultados_cv["Regressão Logística (C=5)"]["f1_macro_media"] else 5
vec = novo_vetorizador()
Xtr_v = vec.fit_transform(X_tr)
modelo = LogisticRegression(C=C_FINAL, max_iter=3000).fit(Xtr_v, y_tr)
classes = list(modelo.classes_)
P_te = modelo.predict_proba(vec.transform(X_te))

# ═════════════ REGRAS DE DECISÃO ═════════════
LIMIAR_ABSTENCAO = 0.40   # confiança mínima para sugerir algo
LIMIAR_URGENCIA = 0.25    # regra de segurança: urgência vence mesmo sem ser a mais provável
i_urg, i_out = classes.index("urgencia"), classes.index("outro")


def decidir(p, texto):
    if sinal_alarme(texto):
        return "urgencia"
    if p[i_urg] >= LIMIAR_URGENCIA:
        return "urgencia"
    top = int(np.argmax(p))
    if classes[top] == "outro" or p[top] < LIMIAR_ABSTENCAO:
        return "outro"
    return classes[top]


pred_bruto = [classes[int(np.argmax(p))] for p in P_te]
pred_final = [decidir(p, t) for p, t in zip(P_te, X_te)]

acc_bruta = accuracy_score(y_te, pred_bruto)
acc_final = accuracy_score(y_te, pred_final)
rel = classification_report(y_te, pred_final, labels=classes, output_dict=True, zero_division=0)

urg_reais = [i for i, c in enumerate(y_te) if c == "urgencia"]
recall_urg = sum(pred_final[i] == "urgencia" for i in urg_reais) / len(urg_reais)
falsos_alarmes = sum(1 for i, c in enumerate(y_te) if c != "urgencia" and pred_final[i] == "urgencia")
urg_perdida = [X_te[i] for i in urg_reais if pred_final[i] != "urgencia"]

print(f"\nModelo final: Regressão Logística C={C_FINAL} | {len(vec.vocabulary_)} features")
print(f"Teste independente ({len(X_te)} frases):")
print(f"  acurácia (só argmax)          {acc_bruta:.3f}")
print(f"  acurácia (com regras)         {acc_final:.3f}")
print(f"  recall de URGÊNCIA            {recall_urg:.3f}  (urgências perdidas: {len(urg_perdida)})")
print(f"  falsos alarmes de urgência    {falsos_alarmes}")
print("\n" + classification_report(y_te, pred_final, labels=classes, zero_division=0))

erros = [(X_te[i], y_te[i], pred_final[i], float(P_te[i].max()))
         for i in range(len(X_te)) if y_te[i] != pred_final[i]]
print("Erros no teste:")
for t, real, prev, conf in erros:
    print(f"  [{real} → {prev} ({conf:.2f})] {t}")


# ═════════════ TESTE B (escrito antes das correções, nunca usado para ajustes) ═════════════
X_b, y_b, _ = ler("dataset_teste_b.csv", False)
vaz_b = [t for t in X_b if " ".join(normalizar(t)) in norm_tr]
P_b = modelo.predict_proba(vec.transform(X_b))
pred_b = [decidir(p, t) for p, t in zip(P_b, X_b)]
acc_b = accuracy_score(y_b, pred_b)
urg_b = [i for i, c in enumerate(y_b) if c == "urgencia"]
recall_urg_b = sum(pred_b[i] == "urgencia" for i in urg_b) / len(urg_b)
alarmes_b = sum(1 for i, c in enumerate(y_b) if c != "urgencia" and pred_b[i] == "urgencia")
print(f"\nTESTE B — {len(X_b)} frases (vazamento: {len(vaz_b)})")
print(f"  acurácia                      {acc_b:.3f}")
print(f"  recall de URGÊNCIA            {recall_urg_b:.3f}")
print(f"  falsos alarmes de urgência    {alarmes_b}")
erros_b = [(X_b[i], y_b[i], pred_b[i], float(P_b[i].max())) for i in range(len(X_b)) if y_b[i] != pred_b[i]]
for t, real, prev, conf in erros_b:
    print(f"  [{real} → {prev} ({conf:.2f})] {t}")


# ═════════════ TESTE C (cego — escrito ANTES dos sinais de alarme) ═════════════
X_c, y_c, _ = ler("dataset_teste_c.csv", False)
vaz_c = [t for t in X_c if " ".join(normalizar(t)) in norm_tr]
P_c = modelo.predict_proba(vec.transform(X_c))
pred_c = [decidir(p, t) for p, t in zip(P_c, X_c)]
acc_c = accuracy_score(y_c, pred_c)
urg_c = [i for i, c in enumerate(y_c) if c == "urgencia"]
recall_urg_c = sum(pred_c[i] == "urgencia" for i in urg_c) / len(urg_c)
alarmes_c = sum(1 for i, c in enumerate(y_c) if c != "urgencia" and pred_c[i] == "urgencia")
rel_c = classification_report(y_c, pred_c, labels=classes, output_dict=True, zero_division=0)
print(f"\nTESTE C (CEGO) — {len(X_c)} frases (vazamento: {len(vaz_c)})")
print(f"  acurácia                      {acc_c:.3f}")
print(f"  recall de URGÊNCIA            {recall_urg_c:.3f}")
print(f"  falsos alarmes de urgência    {alarmes_c}")
erros_c = [(X_c[i], y_c[i], pred_c[i], float(P_c[i].max())) for i in range(len(X_c)) if y_c[i] != pred_c[i]]
for t, real, prev, conf in erros_c:
    print(f"  [{real} → {prev} ({conf:.2f})] {t}")

# Sanidade: sinais de alarme não podem disparar em frases NÃO urgentes do treino
fp_regras = [t for t, c in zip(X_tr, y_tr) if c != "urgencia" and sinal_alarme(t)]
cob_regras = sum(sinal_alarme(t) for t, c in zip(X_tr, y_tr) if c == "urgencia") / y_tr.count("urgencia")
print(f"\nSinais de alarme no treino: cobrem {cob_regras:.1%} das urgências | "
      f"disparos indevidos: {len(fp_regras)} de {len(X_tr) - y_tr.count('urgencia')}")
for t in fp_regras[:10]:
    print("   indevido:", t)

# Matriz de confusão é do teste C (o número honesto)
y_cm, pred_cm, acc_cm = y_c, pred_c, acc_c

# ═════════════ MATRIZ DE CONFUSÃO ═════════════
cm = confusion_matrix(y_cm, pred_cm, labels=classes)
try:
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    fig, ax = plt.subplots(figsize=(7.5, 6.5))
    ax.imshow(cm, cmap="Greens")
    rot = [NOMES[c].split(" /")[0] for c in classes]
    ax.set_xticks(range(len(classes)), rot, rotation=40, ha="right")
    ax.set_yticks(range(len(classes)), rot)
    for i in range(len(classes)):
        for j in range(len(classes)):
            ax.text(j, i, cm[i, j], ha="center", va="center",
                    color="white" if cm[i, j] > cm.max() / 2 else "black", fontsize=11)
    ax.set_xlabel("Previsto pela IA")
    ax.set_ylabel("Categoria correta")
    ax.set_title(f"Matriz de confusão — teste C (cego, {len(X_c)} frases)\nacurácia {acc_cm:.1%}")
    fig.tight_layout()
    fig.savefig("matriz_confusao.png", dpi=150)
    print("\nmatriz_confusao.png salva")
except Exception as e:
    print("matplotlib indisponível:", e)

# ═════════════ EXPORTAÇÃO PARA JAVASCRIPT ═════════════
vocab = vec.vocabulary_
idf = vec.idf_
coef, intercept = modelo.coef_, modelo.intercept_
ordem = sorted(vocab, key=vocab.get)            # features na ordem do índice
exp = {
    "versao": "2.0",
    "algoritmo": f"TF-IDF (palavras + bigramas + char 3-4) + Regressão Logística C={C_FINAL}",
    "classes": classes,
    "nomes": {c: NOMES[c] for c in classes},
    "stopwords": sorted(STOPWORDS),
    "limiarAbstencao": LIMIAR_ABSTENCAO,
    "limiarUrgencia": LIMIAR_URGENCIA,
    "sinaisAlarme": SINAIS_ALARME,
    "sinaisCrise": SINAIS_CRISE,
    "features": ordem,
    "idf": [round(float(idf[vocab[f]]), 5) for f in ordem],
    "coef": [[round(float(coef[k][vocab[f]]), 4) for f in ordem] for k in range(len(classes))],
    "intercept": [round(float(b), 5) for b in intercept],
    "metricas": {"acuracia_teste": round(acc_final, 4), "recall_urgencia": round(recall_urg, 4),
                 "acuracia_teste_b": round(acc_b, 4), "acuracia_teste_c": round(acc_c, 4),
                 "recall_urgencia_teste_c": round(recall_urg_c, 4), "frases_treino": len(X_tr), "frases_teste": len(X_te) + len(X_b)},
}
conteudo_js = ("/* Gerado automaticamente por ia/treinar_modelo.py — NÃO editar à mão. */\n"
               "const MODELO_IA = " + json.dumps(exp, ensure_ascii=False, separators=(",", ":")) + ";\n")
import os
destinos = ["modelo-ia.js"] + (["../frontend/assets/ml/modelo-ia.js"] if os.path.isdir("../frontend/assets/ml") else [])
for d in destinos:
    with open(d, "w", encoding="utf-8") as f:
        f.write(conteudo_js)
print("modelo exportado para:", destinos)

# probabilidades de referência para conferir a implementação JS
X_ref = [x for x in (X_te_orig + X_b + X_c)]
P_ref = modelo.predict_proba(vec.transform(X_ref))
referencia = [{"texto": t, "proba": [round(float(x), 6) for x in p], "decisao": decidir(p, t)}
              for t, p in zip(X_ref, P_ref)]
with open("referencia_python.json", "w", encoding="utf-8") as f:
    json.dump({"classes": classes, "casos": referencia}, f, ensure_ascii=False)

with open("relatorio_metricas.json", "w", encoding="utf-8") as f:
    json.dump({
        "dataset": {"treino": len(X_tr), "teste": len(X_te), "vazamento": len(vazadas),
                    "por_classe_treino": dict(Counter(y_tr)), "por_classe_teste_a": dict(Counter(y_te)), "teste_b": len(X_b), "teste_c": len(X_c)},
        "validacao_cruzada_agrupada": resultados_cv,
        "modelo_final": {"algoritmo": exp["algoritmo"], "n_features": len(ordem)},
        "teste_c_cego": {"frases": len(X_c), "acuracia": acc_c, "recall_urgencia": recall_urg_c,
                         "falsos_alarmes_urgencia": alarmes_c, "por_classe": rel_c,
                         "erros": [{"texto": t, "correto": r, "previsto": p, "confianca": c} for t, r, p, c in erros_c]},
        "sinais_alarme": {"cobertura_urgencias_treino": cob_regras, "disparos_indevidos_treino": len(fp_regras)},
        "teste_b_cego": {"frases": len(X_b), "acuracia": acc_b, "recall_urgencia": recall_urg_b,
                         "falsos_alarmes_urgencia": alarmes_b,
                         "erros": [{"texto": t, "correto": r, "previsto": p, "confianca": c} for t, r, p, c in erros_b]},
        "teste_independente": {"acuracia_argmax": acc_bruta, "acuracia_com_regras": acc_final,
                               "recall_urgencia": recall_urg, "falsos_alarmes_urgencia": falsos_alarmes,
                               "urgencias_perdidas": urg_perdida, "por_classe": rel},
        "matriz_confusao_teste_c": {"classes": classes, "valores": cm.tolist()},
        "erros": [{"texto": t, "correto": r, "previsto": p, "confianca": c} for t, r, p, c in erros],
    }, f, ensure_ascii=False, indent=2)

import os
print(f"modelo-ia.js: {os.path.getsize('modelo-ia.js') / 1024:.0f} KB")


In [ ]:
!python treinar_modelo.py

## 4. Matriz de confusão — teste C (cego)

In [ ]:
from IPython.display import Image, display
display(Image("matriz_confusao.png"))

## 5. Testar frases à mão (usando o modelo exportado — mesma matemática do site)

In [ ]:
import json, re
import numpy as np
exec(open("treinar_modelo.py", encoding="utf-8").read().split("def novo_vetorizador")[0])
src = open("modelo-ia.js", encoding="utf-8").read()
M = json.loads(src[src.index("{"):src.rindex("}") + 1])
idx = {f: i for i, f in enumerate(M["features"])}

def prever(texto):
    t = " ".join(normalizar(texto))
    if any(re.search(r, t) for r in M["sinaisCrise"]):
        return "CRISE → CVV 188 / SAMU 192"
    cont = {}
    for f in analisar(texto):
        if f in idx:
            cont[idx[f]] = cont.get(idx[f], 0) + 1
    x = {j: (1 + np.log(tf)) * M["idf"][j] for j, tf in cont.items()}
    n = np.sqrt(sum(v * v for v in x.values())) or 1
    z = np.array([M["intercept"][k] + sum(v / n * M["coef"][k][j] for j, v in x.items())
                  for k in range(len(M["classes"]))])
    p = np.exp(z - z.max()); p /= p.sum()
    iu = M["classes"].index("urgencia")
    if any(re.search(r, t) for r in M["sinaisAlarme"]) or p[iu] >= M["limiarUrgencia"]:
        return f"URGÊNCIA ({p[iu]:.0%})"
    k = int(p.argmax())
    if M["classes"][k] == "outro" or p[k] < M["limiarAbstencao"]:
        return "(sem sugestão — fora do escopo)"
    return f'{M["nomes"][M["classes"][k]]} ({p[k]:.0%})'

for frase in ["dor de dente forte", "meu bebê tá com febre", "pressão alta", "dor no peito muito forte",
              "que horas abre o posto", "corrimento e coceira", "estou gripado com dor no corpo"]:
    print(f"{frase:35s} → {prever(frase)}")

## 6. Baixar o modelo para o site
Substitua `frontend/assets/ml/modelo-ia.js` pelo arquivo baixado e faça o deploy.

In [ ]:
from google.colab import files
files.download("modelo-ia.js")
files.download("relatorio_metricas.json")

## 7. Gráficos do treinamento
Dataset, comparação de algoritmos, honestidade dos testes, palavras aprendidas, curva de aprendizado.

In [ ]:
%%writefile gerar_graficos.py
"""
gerar_graficos.py — Gráficos do treinamento da IA de triagem (SaúdeMap IA)

Gera, em ia/graficos/, imagens prontas para slides e relatório.
Tudo é calculado a partir dos arquivos reais do projeto:
  dataset_treino.csv, relatorio_metricas.json, comparacao_v1.json,
  ../frontend/assets/ml/modelo-ia.js (ou modelo-ia.js na pasta atual, no Colab)
"""
import csv
import json
import os
import re
from collections import Counter

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.patches import FancyBboxPatch

os.makedirs("graficos", exist_ok=True)

VERDE = "#0A8F82"
VERDE2 = "#02C39A"
ESCURO = "#0B3D3D"
CINZA = "#9AA5A4"
VERMELHO = "#C0392B"
LARANJA = "#E67E22"
plt.rcParams.update({"font.size": 11, "axes.spines.top": False, "axes.spines.right": False,
                     "axes.titleweight": "bold", "axes.titlesize": 14})

NOMES = {"cardio": "Cardiologia", "clinico": "Clínico Geral", "dentista": "Odontologia",
         "gineco": "Ginecologia", "pediatra": "Pediatria", "urgencia": "Urgência",
         "outro": "Fora do escopo"}
COR_CLASSE = {"urgencia": VERMELHO, "outro": CINZA}

R = json.load(open("relatorio_metricas.json", encoding="utf-8"))
caminho_modelo = "../frontend/assets/ml/modelo-ia.js" if os.path.exists("../frontend/assets/ml/modelo-ia.js") else "modelo-ia.js"
src = open(caminho_modelo, encoding="utf-8").read()
M = json.loads(src[src.index("{"):src.rindex("}") + 1])


def salvar(fig, nome):
    fig.tight_layout()
    fig.savefig(f"graficos/{nome}", dpi=160, bbox_inches="tight", facecolor="white")
    plt.close(fig)
    print("  ✓", nome)


# ═══════════ 01 — Dataset por classe ═══════════
linhas = list(csv.DictReader(open("dataset_treino.csv", encoding="utf-8")))
cont = Counter(l["categoria"] for l in linhas)
grupos = {}
for l in linhas:
    grupos.setdefault(l["categoria"], set()).add(l["grupo"])
ordem = sorted(cont, key=cont.get)
fig, ax = plt.subplots(figsize=(10, 5.2))
y = np.arange(len(ordem))
total = [cont[c] for c in ordem]
bases = [len(grupos[c]) for c in ordem]
ax.barh(y, total, color=[COR_CLASSE.get(c, VERDE) for c in ordem], alpha=0.35, label="total com variações (gírias, erros de digitação)")
ax.barh(y, bases, color=[COR_CLASSE.get(c, VERDE) for c in ordem], label="frases-base (sujeito × sintoma, trechos escritos à mão)")
for i, c in enumerate(ordem):
    ax.text(total[i] + 8, i, f"{total[i]}  ({bases[i]} base)", va="center", fontsize=10)
ax.set_yticks(y, [NOMES[c] for c in ordem])
ax.set_xlabel("quantidade de frases")
ax.set_xlim(0, max(total) * 1.28)
ax.set_title(f"Dataset de treino — {len(linhas):,} frases em 7 classes".replace(",", "."))
ax.legend(loc="lower right", frameon=False)
salvar(fig, "01_dataset_por_classe.png")

# ═══════════ 02 — Comparação de algoritmos ═══════════
cv = R["validacao_cruzada_agrupada"]
nomes = list(cv)
acc = [cv[n]["acuracia_media"] * 100 for n in nomes]
dp = [cv[n]["acuracia_desvio"] * 100 for n in nomes]
cores = [VERDE2 if "C=5" in n else (VERDE if "Logística" in n else CINZA) for n in nomes]
fig, ax = plt.subplots(figsize=(10, 5))
b = ax.bar(range(len(nomes)), acc, yerr=dp, capsize=6, color=cores, edgecolor="none")
for i, v in enumerate(acc):
    ax.text(i, v + dp[i] + 0.35, f"{v:.1f}%", ha="center", fontweight="bold")
ax.set_xticks(range(len(nomes)), [n.replace(" (", "\n(") for n in nomes])
ax.set_ylim(88, 100)
ax.set_ylabel("acurácia média (%)")
ax.set_title("Comparação de algoritmos — validação cruzada agrupada (5 folds)")
ax.text(0.99, 0.98, "barra preta = desvio padrão entre os folds\nverde-claro = modelo escolhido (dá probabilidades)",
        transform=ax.transAxes, ha="right", va="top", fontsize=9, color="#555")
salvar(fig, "02_comparacao_algoritmos.png")

# ═══════════ 03 — v1 x v2 ═══════════
if os.path.exists("comparacao_v1.json"):
    v1 = json.load(open("comparacao_v1.json", encoding="utf-8"))["teste_c"]
    v2 = R["teste_c_cego"]
    metricas = ["Acurácia geral", "Urgências detectadas"]
    val_v1 = [v1["acuracia"] * 100, v1["recall_urgencia"] * 100]
    val_v2 = [v2["acuracia"] * 100, v2["recall_urgencia"] * 100]
    fig, ax = plt.subplots(figsize=(9.5, 5.3))
    x = np.arange(2); w = 0.36
    ax.bar(x - w / 2, val_v1, w, color=CINZA, label="v1 — similaridade de cosseno, 82 frases")
    ax.bar(x + w / 2, val_v2, w, color=VERDE, label="v2 — Regressão Logística + sinais de alarme")
    for i in range(2):
        ax.text(x[i] - w / 2, val_v1[i] + 1.5, f"{val_v1[i]:.1f}%", ha="center", fontsize=13, fontweight="bold", color="#555")
        ax.text(x[i] + w / 2, val_v2[i] + 1.5, f"{val_v2[i]:.1f}%", ha="center", fontsize=13, fontweight="bold", color=VERDE)
    ax.set_xticks(x, metricas, fontsize=12)
    ax.set_ylim(0, 115); ax.set_ylabel("%")
    ax.set_title("IA antiga × IA nova — mesmo teste cego (70 frases)")
    ax.legend(loc="upper left", frameon=False)
    salvar(fig, "03_v1_vs_v2.png")


# ═══════════ 04 — Honestidade da medição ═══════════
dados = [("Teste A", R["teste_independente"]["acuracia_com_regras"] * 100, LARANJA,
          "usado para\ncorrigir dados\n→ CONTAMINADO"),
         ("Teste B", R["teste_b_cego"]["acuracia"] * 100, "#F1C40F",
          "regras vieram\ndepois dele\n→ PARCIAL"),
         ("Teste C", R["teste_c_cego"]["acuracia"] * 100, VERDE,
          "nunca usado\npara ajustar\n→ NÚMERO REAL")]
fig, ax = plt.subplots(figsize=(10, 5.6))
for i, (n, v, c, nota) in enumerate(dados):
    ax.bar(i, v, color=c, width=0.6)
    ax.text(i, v + 0.6, f"{v:.1f}%", ha="center", fontsize=15, fontweight="bold")
    ax.text(i, 81.3, nota, ha="center", va="bottom", fontsize=10, fontweight="bold",
            color=ESCURO if c == "#F1C40F" else "white")
ax.axhline(90, ls="--", color=ESCURO, lw=1)
ax.text(2.42, 90.3, "meta 90%", fontsize=9, color=ESCURO, ha="right")
ax.set_xticks(range(3), [d[0] for d in dados], fontsize=12)
ax.set_ylim(80, 102); ax.set_ylabel("acurácia (%)")
ax.set_title("Por que o número honesto é 91,4% e não 98,6%")
ax.text(0.0, -0.12, "atenção: o eixo começa em 80% para destacar a diferença", transform=ax.transAxes,
        fontsize=8.5, color="#777")
salvar(fig, "04_honestidade_dos_testes.png")

# ═══════════ 05 — O que a IA aprendeu (palavras mais fortes por classe) ═══════════
feats = M["features"]; coef = np.array(M["coef"]); classes = M["classes"]
fig, axs = plt.subplots(2, 4, figsize=(15, 7.5))
axs = axs.ravel()
for k, c in enumerate(classes):
    idx = [j for j, f in enumerate(feats) if f.startswith(("w:", "b:"))]
    top = sorted(idx, key=lambda j: coef[k][j], reverse=True)[:8][::-1]
    rot = [feats[j][2:].replace("_", " ") for j in top]
    axs[k].barh(range(len(top)), [coef[k][j] for j in top], color=COR_CLASSE.get(c, VERDE))
    axs[k].set_yticks(range(len(top)), rot, fontsize=10)
    axs[k].set_title(NOMES[c], fontsize=12)
    axs[k].tick_params(axis="x", labelsize=8)
axs[7].axis("off")
axs[7].text(0.0, 0.72, "Cada barra é o PESO que a IA dá\nà palavra. Ninguém escreveu essas\nregras: ela aprendeu dos exemplos.",
            fontsize=10.5, va="center", color=ESCURO)
axs[7].text(0.0, 0.25, "⚠ Vício encontrado: \"tô\", \"agora\",\n\"diz\", \"pai\" NÃO são sintomas.\n"
            "A IA aprendeu também o JEITO como\nas frases de treino foram escritas\n"
            "(shortcut learning). Remédio:\nfrases de pessoas reais.",
            fontsize=10.5, va="center", color=VERMELHO)
fig.suptitle("O que a IA aprendeu — palavras que mais pesam em cada classe", fontsize=15, fontweight="bold")
salvar(fig, "05_o_que_a_ia_aprendeu.png")

# ═══════════ 06 — Como a IA pensa (probabilidades) ═══════════
STOP = set(M["stopwords"]); IDX = {f: i for i, f in enumerate(feats)}
RE_AL = [re.compile(r) for r in M["sinaisAlarme"]]


def norm(t):
    import unicodedata
    t = unicodedata.normalize("NFD", t.lower())
    t = "".join(ch for ch in t if not ("\u0300" <= ch <= "\u036f"))
    return "".join(ch if ("a" <= ch <= "z" or "0" <= ch <= "9") else " " for ch in t).split()


def proba(texto):
    p = [w for w in norm(texto) if w not in STOP]
    f = ["w:" + w for w in p] + ["b:" + p[i] + "_" + p[i + 1] for i in range(len(p) - 1)]
    for w in p:
        s = " " + w + " "
        for n in (3, 4):
            f += ["c:" + s[i:i + n] for i in range(len(s) - n + 1)]
    cnt = Counter(IDX[x] for x in f if x in IDX)
    v = {j: (1 + np.log(tf)) * M["idf"][j] for j, tf in cnt.items()}
    nn = np.sqrt(sum(x * x for x in v.values())) or 1
    z = np.array(M["intercept"]) + sum(coef[:, j] * (x / nn) for j, x in v.items())
    e = np.exp(z - z.max()); return e / e.sum()


# ═══════════ 00 — Pipeline ═══════════
fig, ax = plt.subplots(figsize=(13, 4.2))
ax.set_xlim(0, 13); ax.set_ylim(0, 4.2); ax.axis("off")
etapas = [
    ("Texto\ndigitado", '"meu dente\ntá doendo"', ESCURO),
    ("Normalização", "minúsculas\nsem acento", VERDE),
    ("Features", "palavras + pares\n+ pedaços de letras", VERDE),
    ("TF-IDF", "peso de cada\nfeature", VERDE),
    ("Regressão\nLogística", "probabilidade\npor classe", VERDE),
    ("Camadas de\nsegurança", "alarme · urgência\n· abstenção", VERMELHO),
    ("Resultado", f"Odontologia\n{proba('meu dente tá doendo')[classes.index('dentista')]:.0%}", VERDE2),
]
larg, x0, gap = 1.55, 0.2, 0.28
for i, (titulo, sub, cor) in enumerate(etapas):
    x = x0 + i * (larg + gap)
    ax.add_patch(FancyBboxPatch((x, 1.5), larg, 1.7, boxstyle="round,pad=0.04,rounding_size=0.12",
                                fc=cor, ec="none"))
    ax.text(x + larg / 2, 2.75, titulo, ha="center", va="center", color="white", fontsize=11, fontweight="bold")
    ax.text(x + larg / 2, 1.95, sub, ha="center", va="center", color="white", fontsize=8.5)
    if i < len(etapas) - 1:
        ax.annotate("", xy=(x + larg + gap - 0.02, 2.35), xytext=(x + larg + 0.02, 2.35),
                    arrowprops=dict(arrowstyle="-|>", color=ESCURO, lw=1.6))
ax.text(6.5, 3.85, "Como a IA de triagem processa uma frase", ha="center", fontsize=15, fontweight="bold", color=ESCURO)
ax.text(6.5, 0.75, "Treinado em Python (scikit-learn / Google Colab)  →  pesos exportados  →  executado no navegador",
        ha="center", fontsize=10.5, color="#555")
ax.text(6.5, 0.3, "O texto do cidadão nunca sai do navegador (privacidade / LGPD)", ha="center", fontsize=9.5,
        color=VERDE, style="italic")
salvar(fig, "00_pipeline.png")


frases = ["meu bebê tá com febre", "dor de dnete forte", "dor no peito e falta de ar", "que horas abre o posto"]
fig, axs = plt.subplots(1, 4, figsize=(16, 4.6), sharey=True)
for ax, fr in zip(axs, frases):
    p = proba(fr) * 100
    ax.barh(range(len(classes)), p, color=[COR_CLASSE.get(c, VERDE) for c in classes])
    ax.set_yticks(range(len(classes)), [NOMES[c] for c in classes])
    ax.set_xlim(0, 105)
    for i, v in enumerate(p):
        if v >= 3:
            ax.text(v + 1.5, i, f"{v:.0f}%", va="center", fontsize=9)
    alarme = any(r.search(" ".join(norm(fr))) for r in RE_AL)
    top = classes[int(np.argmax(p))]
    if alarme or p[classes.index("urgencia")] >= M["limiarUrgencia"] * 100:
        dec, cor = "→ URGÊNCIA + SAMU 192", VERMELHO
    elif top == "outro" or p.max() < M["limiarAbstencao"] * 100:
        dec, cor = "→ não sugere nada", CINZA
    else:
        dec, cor = f"→ {NOMES[top]}", VERDE
    ax.set_title(f'"{fr}"\n{dec}', fontsize=11, color=cor)
fig.suptitle("Como a IA pensa — probabilidade de cada classe para 4 frases", fontsize=15, fontweight="bold")
salvar(fig, "06_como_a_ia_pensa.png")

# ═══════════ 07 — Curva de aprendizado ═══════════
try:
    import unicodedata  # noqa
    from sklearn.feature_extraction.text import TfidfVectorizer
    from sklearn.linear_model import LogisticRegression
    from sklearn.metrics import accuracy_score
    from sklearn.model_selection import StratifiedGroupKFold

    def analisar(t):
        p = [w for w in norm(t) if w not in STOP]
        f = ["w:" + w for w in p] + ["b:" + p[i] + "_" + p[i + 1] for i in range(len(p) - 1)]
        for w in p:
            s = " " + w + " "
            for n in (3, 4):
                f += ["c:" + s[i:i + n] for i in range(len(s) - n + 1)]
        return f

    X = np.array([l["texto"] for l in linhas], dtype=object)
    y = np.array([l["categoria"] for l in linhas])
    g = np.array([int(l["grupo"]) for l in linhas])
    rng = np.random.default_rng(0)
    fracs = [0.1, 0.2, 0.35, 0.5, 0.75, 1.0]
    medias, desvios = [], []
    for fr in fracs:
        accs = []
        for tr, va in StratifiedGroupKFold(5, shuffle=True, random_state=1).split(X, y, g):
            grupos_tr = np.unique(g[tr])
            sel = set(rng.choice(grupos_tr, max(7, int(len(grupos_tr) * fr)), replace=False))
            tr2 = [i for i in tr if g[i] in sel]
            vec = TfidfVectorizer(analyzer=analisar, sublinear_tf=True, min_df=2)
            m = LogisticRegression(C=5, max_iter=3000).fit(vec.fit_transform(X[tr2]), y[tr2])
            accs.append(accuracy_score(y[va], m.predict(vec.transform(X[va]))))
        medias.append(np.mean(accs) * 100); desvios.append(np.std(accs) * 100)
        print(f"    curva: {fr:.0%} dos dados → {medias[-1]:.1f}%")
    n_fr = [int(len(X) * 0.8 * f) for f in fracs]  # cada fold treina com ~80% dos dados
    fig, ax = plt.subplots(figsize=(10, 5))
    ax.plot(n_fr, medias, "o-", color=VERDE, lw=2.5, ms=8)
    ax.fill_between(n_fr, np.array(medias) - desvios, np.array(medias) + desvios, color=VERDE, alpha=0.15)
    for x_, m_ in zip(n_fr, medias):
        ax.text(x_, m_ + 1.2, f"{m_:.1f}%", ha="center", fontsize=9.5)
    ax.set_xlabel("frases de treino"); ax.set_ylabel("acurácia na validação (%)")
    ax.set_ylim(min(medias) - 8, 100)
    ax.set_title("Curva de aprendizado — mais dados, IA melhor")
    ax.text(0.98, 0.06, "a curva ainda sobe no final:\nmais frases (principalmente REAIS)\nainda devem melhorar a IA",
            transform=ax.transAxes, ha="right", fontsize=9.5, color="#555")
    salvar(fig, "07_curva_de_aprendizado.png")
except Exception as e:
    print("  curva de aprendizado ignorada:", e)

# ═══════════ 08 — Matriz de confusão (cópia) ═══════════
if os.path.exists("matriz_confusao.png"):
    import shutil
    shutil.copy("matriz_confusao.png", "graficos/08_matriz_confusao.png")
    print("  ✓ 08_matriz_confusao.png")

print("\nGráficos em ia/graficos/")


In [ ]:
!python gerar_graficos.py

In [ ]:
import glob
from IPython.display import Image, display
for f in sorted(glob.glob("graficos/*.png")):
    print(f); display(Image(f))